## 1. Environment Setup & Imports

In this initial step, we set up our workspace by:
* **Importing core libraries:** Loading the necessary tools for data manipulation (`pandas`), splitting (`train_test_split`), imputation, and feature scaling/encoding.
* **Configuring project paths:** Dynamically adding the project root to the system path. This ensures Jupyter can locate and import our custom directory variables (`RAW_DATA_DIR`, `PROCESSED_DATA_DIR`) from the `src` folder without throwing a `ModuleNotFoundError`.

In [26]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder


PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR

## 2. Load Data & Encode Target Variable

In this step, we bring in our raw dataset and prepare our target variable for machine learning:
* **Loading and Cleaning:** We read the `train.csv` file and immediately drop the `id` column. Unique identifiers don't contain predictive patterns and can confuse the model.
* **Encoding the Target:** Machine learning algorithms require numbers, not text. We use `LabelEncoder` to convert our target variable (`Personality`) from text labels into numeric values (e.g., 0, 1, 2).
* **Sanity Check:** We print the exact mapping dictionary so we know exactly which number represents which personality type, followed by a quick preview of the dataframe to ensure everything looks correct.

In [21]:

file_path = RAW_DATA_DIR / "train.csv"
df = pd.read_csv(file_path)
df_clean = df.drop(columns=['id'])

le = LabelEncoder()
df_clean['Personality'] = le.fit_transform(df_clean['Personality'])

print("Personality Mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df_clean.head())


Personality Mapping: {'Extrovert': np.int64(0), 'Introvert': np.int64(1)}
   Time_spent_Alone Stage_fear  Social_event_attendance  Going_outside  \
0               0.0         No                      6.0            4.0   
1               1.0         No                      7.0            3.0   
2               6.0        Yes                      1.0            0.0   
3               3.0         No                      7.0            3.0   
4               1.0         No                      4.0            4.0   

  Drained_after_socializing  Friends_circle_size  Post_frequency  Personality  
0                        No                 15.0             5.0            0  
1                        No                 10.0             8.0            0  
2                       NaN                  3.0             0.0            1  
3                        No                 11.0             5.0            0  
4                        No                 13.0             NaN            0  


## 3. Train-Test Split (Preventing Data Leakage)

Before we apply any statistical transformations (like imputation or target encoding), we **must** split our dataset. If we calculate means, medians, or category probabilities on the entire dataset *before* splitting, information from the test set "leaks" into the training set, leading to overly optimistic and inaccurate model performance.

* **Isolate Features vs. Target:** We separate our predicting features (`X`) from the target variable we want to predict (`y`, Personality).
* **The Split:** We divide the data into a training set (70%) to teach the model, and a testing set (30%) to evaluate it. We set `random_state=42` to ensure our split is reproducible every time we run the notebook.

In [14]:
X = df_clean.drop('Personality', axis=1)
y = df_clean['Personality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


## 4. Impute Missing Values

Real-world data is rarely perfect. Here, we fill in (impute) any missing values in our dataset to ensure our machine learning models don't crash. We handle different types of data with different strategies:

* **Categorical Data (`most_frequent`):** For text or binary columns like `Stage_fear`, we replace missing values with the most common answer (the mode).
* **Numerical Data (`median`):** For continuous numbers, we use the median. The median is generally preferred over the average (mean) because it is less affected by extreme outliers.
* **Preventing Data Leakage:** Notice that we use `.fit_transform()` on the training data, but only `.transform()` on the test data. We must calculate the median/mode *only* from the training set, and then apply those exact same fill values to the test set to simulate real-world, unseen data.

In [22]:
cat_cols = ['Stage_fear', 'Drained_after_socializing']
num_cols = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 'Friends_circle_size', 'Post_frequency']

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

num_imputer = SimpleImputer(strategy='median')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
print(X_train.head())

       Time_spent_Alone  Stage_fear  Social_event_attendance  Going_outside  \
5905          -0.708322     0.07121                 1.413231      -0.019093   
16801         -1.049865     0.07121                 1.787227       0.486129   
6395          -0.025235     0.07121                 1.039235       0.991350   
14183         -0.366779     0.07121                 0.291242       0.486129   
9981          -1.049865     0.07121                 0.291242       1.496572   

       Drained_after_socializing  Friends_circle_size  Post_frequency  
5905                    0.068836            -0.978813        0.733313  
16801                   0.068836             0.239746        1.454747  
6395                    0.068836             1.214593       -0.348838  
14183                   0.068836            -0.003966        0.011879  
9981                    0.068836            -0.003966        0.011879  


## 5. Feature Encoding and Scaling

Now that our missing data is handled, we need to mathematically prepare the values so our machine learning model can process them optimally without bias.

* **Target Encoding (`cat_cols`):** We convert our text-based categories ("Yes"/"No") into numerical probabilities based on their relationship with the target variable (`y_train`). Notice we **only fit this on the training data** to ensure the model doesn't "peek" at the test set's answers (preventing data leakage).
* **Feature Scaling (`num_cols`):** Machine learning models can be easily confused if one feature has values in the thousands while another is in the single digits. `StandardScaler` levels the playing field by converting all continuous numbers into "z-scores" (where the average becomes 0). Just like our imputers, we calculate the mathematical scaling rules on the training set, and apply those exact same rules to the test set.

In [24]:
encoder = TargetEncoder(cols=cat_cols)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols], y_train)
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

display(X_train.head())

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
5905,-0.708322,0.07121,1.413231,-0.019093,0.068836,-0.978813,0.733313
16801,-1.049865,0.07121,1.787227,0.486129,0.068836,0.239746,1.454747
6395,-0.025235,0.07121,1.039235,0.991350,0.068836,1.214593,-0.348838
14183,-0.366779,0.07121,0.291242,0.486129,0.068836,-0.003966,0.011879
9981,-1.049865,0.07121,0.291242,1.496572,0.068836,-0.003966,0.011879


## 6. Save Processed Data

With our data fully cleaned, encoded, and scaled, the final step is to save it. By exporting these final dataframes to our `processed` directory, we create a clean, ready-to-use checkpoint.

* **Creating the Directory:** We use `.mkdir(parents=True, exist_ok=True)` to safely ensure the `data/processed/` folder exists before we try to save anything into it.
* **Exporting to CSV:** We save our training and testing features (`X`) and targets (`y`) as separate files. Setting `index=False` ensures we don't accidentally save the pandas row numbers as a new, meaningless column that could confuse our future models.

Now, our modeling scripts can simply load this prepared data instantly!

In [25]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)

print("Data saved successfully!")

Data saved successfully!
